# LifeMate trên Colab

Chọn GPU trong **Runtime → Change runtime type**, rồi bấm **Run all**. Cấp quyền Drive khi Colab hỏi. Notebook chuẩn bị môi trường để chạy LifeMate.


In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

ZAI = Path("/content/drive/MyDrive/ZAI")
DATA_DIR = ZAI / "data"
MODELS_DIR = ZAI / "models"
RUNS_DIR = ZAI / "runs"
REPO_DIR = Path("/content/ZaloAI")
REPO_URL = "https://github.com/UIT-26521197/ZaloAI.git"
BRANCH = "codex/lifemate-baseline"



## Đồng bộ code và cài thư viện

Sau khi push code từ máy lên nhánh trong `BRANCH`, chạy lại cell này trên Colab. Nếu repo private, xác thực GitHub trong runtime trước khi clone.


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    git = ["git", "-C", str(REPO_DIR)]
    remote = subprocess.check_output(git + ["remote", "get-url", "origin"], text=True).strip()
    branch = subprocess.check_output(git + ["branch", "--show-current"], text=True).strip()
    if remote != REPO_URL or branch != BRANCH:
        raise RuntimeError(f"{REPO_DIR} đang dùng {remote} / {branch}. Hãy sửa REPO_DIR hoặc BRANCH.")
    subprocess.run(git + ["pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")],
    check=True,
)

# Dùng tiến trình Python mới để tránh import lại các module cũ trong kernel.
subprocess.run([sys.executable, "-c", """
import numpy
import torch
import transformers
import huggingface_hub
from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoModelForSpeechSeq2Seq, AutoProcessor, AutoTokenizer

if Version(torch.__version__) < Version("2.6"):
    raise RuntimeError("Cần PyTorch 2.6 trở lên.")
if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU trong Runtime → Change runtime type.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Môi trường LifeMate đã sẵn sàng.")
"""], check=True)
